In [ ]:
!pip install gradio transformers torch fpdf

In [ ]:
import gradio as gr
from transformers import pipeline
from fpdf import FPDF

# AI model
generator = pipeline("text-generation", model="Salesforce/codegen-350M-mono")

history = []

schema = """
Tables:
users(id, name, email, signup_date)
transactions(id, user_id, amount, date)
"""

def generate_sql(question):

    prompt = f"""
Convert the natural language request into SQL query.

Database schema:
{schema}

Request: {question}

SQL Query:
"""

    result = generator(prompt, max_length=120)

    text = result[0]["generated_text"]
    sql = text.split("SQL Query:")[-1].strip()

    history.append(sql)

    return "\n".join(history)


def clear_history():
    history.clear()
    return ""


def download_txt():
    with open("query.txt","w") as f:
        for q in history:
            f.write(q+"\n")
    return "query.txt"


def download_pdf():
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)

    for q in history:
        pdf.cell(200,10,txt=q,ln=True)

    pdf.output("query.pdf")

    return "query.pdf"


with gr.Blocks() as demo:

    gr.Markdown("# Query Console")

    with gr.Row():

        with gr.Column(scale=3):
            console = gr.Textbox(lines=15,label="Query Console")

            inp = gr.Textbox(label="Natural Language Request")

            btn = gr.Button("Generate SQL")

        with gr.Column(scale=1):

            clear_btn = gr.Button("Clear History")

            gr.Markdown("### Export")

            pdf_btn = gr.Button("Download PDF")
            txt_btn = gr.Button("Download TXT")

            pdf_file = gr.File(label="PDF File")
            txt_file = gr.File(label="TXT File")

    btn.click(generate_sql,inputs=inp,outputs=console)

    clear_btn.click(clear_history,outputs=console)

    pdf_btn.click(download_pdf,outputs=pdf_file)

    txt_btn.click(download_txt,outputs=txt_file)

demo.launch()